# Unidad 1 — Tratamiento de datos

Esta unidad trabaja la **estadística descriptiva**: cómo resumir, ubicar y comunicar
una muestra de datos antes de hablar de probabilidad. Trabajamos sobre tres miradas:

- **Tendencia central** (¿dónde está el centro?): media, mediana.
- **Dispersión** (¿qué tan apretados están los datos?): varianza, desvío, IQR.
- **Posición y outliers** (¿hay puntos raros?): cuartiles, regla de Tukey.

El código vive en `src/descriptive/` y `src/visualization/`; este notebook sólo lo orquesta.


## Importaciones

El paquete `probabilidad` se instala editable al correr `uv sync`. No hay `sys.path`.


In [ ]:
import numpy as np
import pandas as pd

from core import Observations, Settings
from descriptive import (
    FrequencyTableInput,
    build_frequency_table,
    detect_outliers_tukey,
    standardize_observations,
    summarize_observations,
)
from exercises import NumericAnswerInput, verify_numeric_answer
from visualization import (
    DescriptiveSummaryChartInput,
    FrequencyChartInput,
    chart_descriptive_summary,
    chart_frequency_table,
)
from widgets import DescriptiveExplorerInput, build_descriptive_explorer

In [ ]:
settings = Settings()

## Concreto (C de CPA): una muestra que podemos imaginar

Pensemos en los **tiempos de espera (minutos) en una caja** durante 80 turnos.
La generamos a partir de una Normal para tener un caso prolijo y reproducible.


In [ ]:
rng = np.random.default_rng(settings.random_seed)
raw_waiting_times = rng.normal(loc=4.0, scale=1.2, size=80).clip(min=0.0)
waiting_times = Observations.validate(pd.DataFrame({"value": raw_waiting_times}))
waiting_times.head()

## Pictórico (P de CPA): histograma con marca de clase y ojiva

Antes de cualquier número, **dibujamos**. El histograma muestra la *forma* y la ojiva
(la frecuencia relativa acumulada) responde "¿qué porción de datos está por debajo de _x_?".


In [ ]:
frequency_table = build_frequency_table(FrequencyTableInput(observations=waiting_times, bin_count=10))
chart_frequency_table(FrequencyChartInput(frequency_table=frequency_table, settings=settings))

## Abstracto (A de CPA): resumen numérico

El resumen condensa la muestra en un objeto Pydantic — sin tablas sueltas — para que
el resto del notebook lo pueda consumir de forma segura.


In [ ]:
summary = summarize_observations(waiting_times)
summary

### Intuición: media vs. mediana

Cuando la distribución es **simétrica** (como acá), media y mediana están casi pegadas.
Cuando hay **cola larga**, la media se mueve hacia la cola y la mediana se queda en el centro.
Por eso la **mediana** es el resumen "robusto".


In [ ]:
chart_descriptive_summary(
    DescriptiveSummaryChartInput(observations=waiting_times, statistics=summary, settings=settings)
)

## Detección de outliers (regla de Tukey)

Un punto es outlier si cae fuera de $[Q_1 - 1{,}5\,IQR,\ Q_3 + 1{,}5\,IQR]$.
Es la misma regla que dibuja el *boxplot*: por eso lo usamos para pensar en outliers.


In [ ]:
outlier_report = detect_outliers_tukey(waiting_times)
outlier_report

## Posición: $z$-score

El $z$-score traduce una observación a la pregunta: **¿a cuántos desvíos del promedio está?**.
Es la *moneda común* de la unidad: nos prepara para Unidad 3 (estandarización de la Normal).

$$ z_i = \frac{x_i - \bar{x}}{s} $$


In [ ]:
standardized = standardize_observations(waiting_times)

## Exploración interactiva

Movés μ, σ, n y la cantidad de bins. El histograma, la ojiva y el boxplot se recalculan en
vivo. Probá empujar σ a valores grandes: ¿qué pasa con el IQR vs. el rango?


In [ ]:
build_descriptive_explorer(DescriptiveExplorerInput(settings=settings))

## Ejercicio 1 — Media de una muestra

**Enunciado.** Sea la muestra $\{2, 4, 4, 4, 5, 5, 7, 9\}$. Calcular la media muestral.

**Idea.** Sumar todos los valores y dividir por $n = 8$.

$$ \bar{x} = \frac{2 + 4 + 4 + 4 + 5 + 5 + 7 + 9}{8} = \frac{40}{8} = 5 $$


In [ ]:
exercise_sample = Observations.validate(pd.DataFrame({"value": [2.0, 4.0, 4.0, 4.0, 5.0, 5.0, 7.0, 9.0]}))
expected_mean = summarize_observations(exercise_sample).location.mean

student_answer = 5.0
verify_numeric_answer(NumericAnswerInput(student_answer=student_answer, expected_answer=expected_mean))

## Ejercicio 2 — Desvío estándar muestral

**Enunciado.** Para la misma muestra, calcular el **desvío estándar muestral** $s$
(divisor $n-1$).

$$ s = \sqrt{\frac{1}{n-1}\sum_{i=1}^{n}(x_i - \bar{x})^2} $$

Con $\bar{x} = 5$:

$$ s = \sqrt{\frac{(2-5)^2 + 3(4-5)^2 + 2(5-5)^2 + (7-5)^2 + (9-5)^2}{7}} = \sqrt{\frac{32}{7}} \approx 2{,}138 $$


In [ ]:
expected_standard_deviation = summarize_observations(exercise_sample).dispersion.sample_standard_deviation
student_answer = 2.138
verify_numeric_answer(
    NumericAnswerInput(
        student_answer=student_answer,
        expected_answer=expected_standard_deviation,
        absolute_tolerance=1e-2,
        relative_tolerance=1e-2,
    )
)

## Para llevarse

- **Resumir = ubicar + dispersar + posicionar**. No alcanza con uno solo.
- La **mediana** y el **IQR** son robustos a outliers; la **media** y $s$ no.
- El $z$-score es la **misma idea** que vamos a usar para la Normal estándar en Unidad 3.
